# Synaptic Routing Architecture (SRA)
## 11: 시냅스의 동적 삭제(Hot-Swap의 취소·특정 도메인의 퍼지)

이 노트북은 SRA의 기능인 "시냅스 삭제"를 보여줍니다. 구체적으로 다음 두 가지 시나리오를 확인합니다.
1. **플러그인 제거(`pop_synapses`)**: 나중에 추가(Hot-Swap)한 시냅스를 끝에서 물리적으로 제거하고 추가 전 상태로 복원합니다.
2. **특정 도메인 퍼지(`clear_synapses`)**: 초기 기본 모델에서 특정 기능(예: Math)만 추출하고 다른 사람과 공유하지 않는 시냅스를 안전하게 '제로 클리어(빈 슬롯화)'합니다.

※ 이 노트북은 GPU가 없어도 실행 가능합니다.

In [ ]:
# 0. 環境セットアップ (Google Colab用)
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from sra_language_models import MoESRALanguageModel
from sra_experiment import find_unshared_synapses
import torch
import tiktoken


### 1. 플러그인 분리(`pop_synapses`)
우선은, 소규모의 모델을 구축해, 동적으로 시냅스를 추가한 후, 그것을 `pop_synapses` 로 떼어내는 동작을 확인합니다.

In [ ]:
# モデルの初期化
dim = 128
layers = 2
num_synapses = 4
k = 2
syn_hidden = 256
vocab_size = 1000

print("=== プラグインの取り外しデモ ===")
model = MoESRALanguageModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"[初期状態] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの追加 (Hot-Swap)
print("\nプラグインとしてシナプスを 2つ 追加します...")
model.add_synapses(2, freeze_base=True)
print(f"[追加後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの取り外し (Undo Hot-Swap)
print("\n追加したシナプスを 1つ 取り外します...")
model.pop_synapses(1)
print(f"[取り外し後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

이와 같이, `pop_synapses(N)` 를 호출하는 것으로, 말미로부터 N개의 시냅스 텐서가 물리적으로 삭제되어 모델의 용량을 복원할 수 있습니다.

### 2. 특정 도메인 제거(`clear_synapses` 및 `find_unshared_synapses`)
다음으로, 복수 도메인을 학습한 베이스 모델중에서, 「특정 도메인(여기에서는 만일 `math`)에서 밖에 사용되지 않는 불필요한 시냅스」를 자동 검출해, 제로 클리어에 의해 안전하게 무효화하는 프로세스를 실증합니다.

In [ ]:
import random
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"

# 仮のデータセットとバッチ関数の準備（デモ用）
# 本物のデータセットを読み込む（ランダムノイズだとルーターが専門化できないため）
import os
import sys
domains = ["text", "code", "math"]
data = {}
base_dir = "." if 'google.colab' in sys.modules else ".."
for d in domains:
    path = f"{base_dir}/data/lang/{d}.txt"
    with open(path, "r", encoding="utf-8") as f:
        text = (f.read() + "\n") * 5
    tokens = tokenizer.encode(text, allowed_special="all")
    data[d] = torch.tensor(tokens, dtype=torch.long)
def dummy_get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    for i in range(batch_size):
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x.to(device), y.to(device)

# もう少し大きめのモデルを準備
def get_multidomain_batch_with_mask(data_dict, batch_size, seq_len):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    mask = torch.zeros((batch_size, 4), dtype=torch.bool)
    domain_map = {'text': 0, 'code': 1, 'math': 2}
    
    for i in range(batch_size):
        d = random.choice(list(data_dict.keys()))
        d_data = data_dict[d]
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
        mask[i, domain_map[d]] = True
        
    return x.to(device), y.to(device), mask.to(device)

multi_model = MoESRALanguageModel(vocab_size, dim=128, layers=2, num_synapses=16, k=2, syn_hidden=256).to(device)

첫째, 노트북 09와 마찬가지로 모델에 여러 도메인을 수천 단계 학습하고 도메인 특화 시냅스를 자연스럽게 만듭니다.
(※Colab의 CPU 환경에서는 몇 분 정도 걸립니다)

In [ ]:
# 学習パラメータ
batch_size = 32
steps = 2500
lr = 1e-3
opt = torch.optim.AdamW(multi_model.parameters(), lr=lr)

multi_model.train()
for step in range(1, steps + 1):
    # パージデモを確実にするため、学習時にメタデータマスクを与えてドメインを完全分離させます
    x, y, mask = get_multidomain_batch_with_mask(data, batch_size, 32)
    
    logits, router_logits = multi_model(x, dense=False, allowed_synapses_mask=mask)
    
    ce_loss = torch.nn.functional.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss = ce_loss
    
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(multi_model.parameters(), 1.0)
    opt.step()
    
    if step % 500 == 0:
        print(f"Step {step}/{steps} | Loss: {loss.item():.4f}")


学習が完了しました。削除を行う**前**に、各ドメインの言語モデリング性能（Loss）を評価して、モデルが正しく機能していることを確認します。

In [ ]:
import torch.nn.functional as F

def evaluate_domain(model, domain, batches=10):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for _ in range(batches):
            x, y = dummy_get_batch(data, 32, 32, domain)
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = torch.nn.functional.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            total_loss += loss.item()
    return total_loss / batches

print("=== [削除前] ドメイン別 Loss ===")
pre_losses = {}
for d in domains:
    loss = evaluate_domain(multi_model, d)
    pre_losses[d] = loss
    print(f"{d.upper()} Loss: {loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sra_experiment import usage_stats

multi_model.eval()
domain_usages = {}

with torch.no_grad():
    for d in domains:
        # 評価用に数バッチ回して使用状況を集計
        total_usage = None
        for _ in range(5):
            x, y = dummy_get_batch(data, batch_size, 32, d)
            x, y = x.to(device), y.to(device)
            _, router_logits = multi_model(x)
            
            u = usage_stats(router_logits) # shape: (num_synapses,)
            total_usage = u if total_usage is None else total_usage + u
            
        # 正規化して保存
        domain_usages[d] = (total_usage / total_usage.sum()).cpu().numpy()

# ヒートマップ描画 (ドメイン x シナプス)
usage_matrix = np.array([domain_usages[d] for d in domains])

plt.figure(figsize=(10, 4))
sns.heatmap(usage_matrix, cmap="Blues", annot=True, fmt=".2f", yticklabels=domains)
plt.title("Synapse Usage per Domain")
plt.xlabel("Synapse ID")
plt.ylabel("Domain")
plt.show()

In [ ]:
print("=== 特定ドメインのパージデモ ===")
print("Mathドメインでのみ使用され、他(Text, Code)と共用されていないシナプスを検索します...")

# ユーティリティを用いて対象シナプスを特定
unshared_synapses = find_unshared_synapses(
    model=multi_model, 
    data_dict=data, 
    target_domain="math", 
    other_domains=["text", "code"], 
    get_batch_func=dummy_get_batch,
    max_seq_len=32,
    threshold=0.20
)

if len(unshared_synapses) == 0:
    # 400ステップでは完全に分離しきれない場合があるため、最もMathで利用されたシナプスを強制抽出
    from sra_experiment import usage_stats
    multi_model.eval()
    math_usages = []
    with torch.no_grad():
        for _ in range(5):
            x, y = dummy_get_batch(data, 16, 32, "math")
            x, y = x.to(device), y.to(device)
            _, r_logits = multi_model(x)
            math_usages.append(usage_stats(r_logits))
        target_idx = sum(math_usages).argmax().item()
        unshared_synapses = [target_idx]

print(f"\n抽出されたMath専用のシナプスインデックス: {unshared_synapses}")


제대로 학습된 모델에서 훌륭하게 Math 전용 시냅스가 추출되었습니다!


In [ ]:
target_idx = unshared_synapses[0]

# クリア前の重みのノルムを確認
pre_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
pre_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア前] シナプス {target_idx} の Router埋め込みノルム: {pre_emb_norm:.4f}, W1ノルム: {pre_w1_norm:.4f}")

# ゼロクリア実行
multi_model.clear_synapses([target_idx])
print("\nゼロクリアを実行しました。\n")

# クリア後の重みのノルムを確認
post_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
post_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア後] シナプス {target_idx} の Router埋め込みノルム: {post_emb_norm:.4f}, W1ノルム: {post_w1_norm:.4f}")


### 3. パージ後の機能評価（Zero Forgettingの証明）

Math専用のシナプスを無効化しました。最後に、もう一度各ドメインの性能を評価します。
**MathのLossだけが悪化し、TextとCodeのLossは削除前と完全に一致（Zero Forgetting）**していることを確認してください。

In [ ]:
print("=== [削除後] ドメイン別 Loss ===")
post_losses = {}
for d in domains:
    loss = evaluate_domain(multi_model, d)
    post_losses[d] = loss
    
    diff = loss - pre_losses[d]
    # わずかなノイズを考慮し、0.005以上悪化した場合を破壊と判定
    status = "❌ 破壊 (Loss悪化)" if diff > 0.005 else "✅ 無傷 (Zero Forgetting)"
    print(f"{d.upper()} Loss: {loss:.4f} (差分: {diff:+.4f}) -> {status}")


### 결론
`clear_synapses` 는 지정된 인덱스의 가중치를 물리적으로 삭제하지 않고 `0.0` 에 클리어 합니다.
이렇게 하면 다른 시냅스의 인덱스(ID)가 어긋나는 것을 방지하고 기존 메타데이터 마스크와의 호환성을 유지하면서 불필요한 계산 경로를 완전히 비활성화할 수 있습니다. 빈 슬롯이 된 인덱스에는 나중에 새 플러그인을 덮어 쓸 수 있습니다.